# 00 · Acquire the raw data (optional)
**Goal:** download the 27 raw sequencing runs (`prefetch`, then `vdb-validate`). **Input:** `metadata/samples.csv`. **Output:** `raw_data/sra/<SRR>/<SRR>.sra` (~54 GB, not in git). **Kernel:** Python 3 · **Time:** 30–60 min.

Only needed for local runs and the adapter check in notebook 02; Euler downloads its own copy (notebook 01). Everything is resumable: finished runs are skipped. Study and samples: [docs/data.md](../docs/data.md); terms: [docs/glossary.md](../docs/glossary.md).

## 1 · The sample table
One row per run; `group` is the analysis group.

In [1]:
# helpers.py holds the pipeline functions (download, QC, trimming, kallisto); h.status() lists, per run, whether it is downloaded / quantified
import sys; sys.path.insert(0, '../scripts')
import helpers as h
runs = h.load_runs()
h.status(runs)

,group,sra_GB,quantified,trimmed
SRR,,,,
SRR21355401,aging 3mo,1.67,True,True
SRR21355396,aging 3mo,3.03,True,True
SRR21355392,aging 3mo,1.88,True,True
SRR21355388,aging 3mo,1.94,True,True
SRR21355398,aging 18mo,0.78,True,True
SRR21355394,aging 18mo,0.95,True,True
SRR21355390,aging 18mo,3.23,True,True
SRR21355386,aging 18mo,1.13,True,True
SRR21355397,aging 21mo,1.66,True,True


In [2]:
# the 7 analysis groups: number of mice, sex (M/F) and age
runs.groupby('group', sort=False).agg(mice=('mouse_id', 'size'), sex=('sex', lambda s: ''.join(sorted(x[0].upper() for x in s))), age_months=('age_months', 'first'))

,mice,sex,age_months
group,,,
aging 3mo,4,FFMM,3
aging 18mo,4,FFMM,18
aging 21mo,4,FFMM,21
AL,4,FFFF,19
aDR,5,FFFFF,19
PBS,3,MMM,19
YMP,3,MMM,19


## 2 · Download (~54 GB)
Progress bar in GB; per-run logs in `results/logs/<SRR>.log`. The `.sra` files are kept; the FASTQ files made from them (notebook 02) are temporary.

In [3]:
# prefetch every run that is not yet on disk, then validate it
h.download_all(runs)

all 27 runs present


In [4]:
# check: number of runs present and total size
s = h.status(runs); print(f"{s.sra_GB.notna().sum()}/27 runs, {s.sra_GB.sum():.1f} GB")
s

27/27 runs, 54.2 GB


,group,sra_GB,quantified,trimmed
SRR,,,,
SRR21355401,aging 3mo,1.67,True,True
SRR21355396,aging 3mo,3.03,True,True
SRR21355392,aging 3mo,1.88,True,True
SRR21355388,aging 3mo,1.94,True,True
SRR21355398,aging 18mo,0.78,True,True
SRR21355394,aging 18mo,0.95,True,True
SRR21355390,aging 18mo,3.23,True,True
SRR21355386,aging 18mo,1.13,True,True
SRR21355397,aging 21mo,1.66,True,True


## 3 · Verify integrity
Checksums of all 27 files (a few minutes); an empty list means every file is intact.

In [5]:
# returns the runs whose .sra file fails validation
h.validate_all(runs)

validate:   0%|          | 0/27 [00:00<?, ?it/s]

[]